In [4]:
!pip install -q gradio requests pandas

In [16]:

# WEATHER FORECASTING APP
# Google Colab Version
# Python + Gradio + OpenWeather API
import requests
import gradio as gr
import pandas as pd
from datetime import datetime

# 1. OPENWEATHER API KEY

API_KEY = "Api Key"
CURRENT_WEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"
FORECAST_URL = "https://api.openweathermap.org/data/2.5/forecast"

# 2. GET WEATHER DATA

def get_weather(city):
    if not city or city.strip() == "":
        return ("Please enter a city name.","","","","","","","",pd.DataFrame())
    if API_KEY == "YOUR_OPENWEATHER_API_KEY":
        return ("API Key Missing","Please add your OpenWeather API key in the code.","","","","","","",pd.DataFrame())
    city = city.strip()
    try:
        # CURRENT WEATHER
        params = {
            "q": city,
            "appid": API_KEY,
            "units": "metric"
        }
        response = requests.get(
            CURRENT_WEATHER_URL,
            params=params,
            timeout=10
        )
        if response.status_code == 401:
            return ("Invalid API Key","Please check your OpenWeather API key.","","","","","","",pd.DataFrame())

        if response.status_code == 404:
            return (
                "City Not Found",
                "Please enter a valid city name.",
                "",
                "",
                "",
                "",
                "",
                "",
                pd.DataFrame()
            )

        response.raise_for_status()

        data = response.json()


        # EXTRACT CURRENT WEATHER


        city_name = data["name"]
        country = data["sys"]["country"]

        temperature = data["main"]["temp"]
        feels_like = data["main"]["feels_like"]

        humidity = data["main"]["humidity"]
        pressure = data["main"]["pressure"]

        wind_speed = data["wind"]["speed"]

        condition = data["weather"][0]["main"]
        description = data["weather"][0]["description"]

        icon_code = data["weather"][0]["icon"]

        icon_url = (
            f"https://openweathermap.org/img/wn/"
            f"{icon_code}@2x.png"
        )


        # SUNRISE / SUNSET


        sunrise_timestamp = data["sys"]["sunrise"]
        sunset_timestamp = data["sys"]["sunset"]

        sunrise = datetime.fromtimestamp(
            sunrise_timestamp
        ).strftime("%I:%M %p")

        sunset = datetime.fromtimestamp(
            sunset_timestamp
        ).strftime("%I:%M %p")

        # CURRENT WEATHER DISPLAY


        location = f"## {city_name}, {country}"

        current_weather = f"""
###  {temperature:.1f}°C

**{condition}** — {description.title()}

![Weather Icon]({icon_url})

**Feels Like:** {feels_like:.1f}°C
"""

        humidity_text = f" **Humidity:** {humidity}%"

        wind_text = f" **Wind Speed:** {wind_speed:.1f} m/s"

        pressure_text = f" **Pressure:** {pressure} hPa"

        sunrise_text = f" **Sunrise:** {sunrise}"

        sunset_text = f" **Sunset:** {sunset}"


        # 5-DAY FORECAST

        forecast_params = {
            "q": city,
            "appid": API_KEY,
            "units": "metric"
        }

        forecast_response = requests.get(
            FORECAST_URL,
            params=forecast_params,
            timeout=10
        )

        forecast_response.raise_for_status()

        forecast_data = forecast_response.json()

        forecast_list = forecast_data["list"]

        # SELECT ONE FORECAST PER DAY

        daily_data = {}

        for item in forecast_list:

            date = item["dt_txt"].split(" ")[0]

            if date not in daily_data:
                daily_data[date] = item


        days = list(daily_data.values())[:5]


        # CREATE FORECAST TABLE


        forecast_rows = []

        for day in days:

            date_object = datetime.strptime(
                day["dt_txt"],
                "%Y-%m-%d %H:%M:%S"
            )

            date_display = date_object.strftime(
                "%a, %d %b"
            )

            temp = day["main"]["temp"]

            feels = day["main"]["feels_like"]

            humidity_day = day["main"]["humidity"]

            wind = day["wind"]["speed"]

            weather = day["weather"][0]["main"]

            description_day = day["weather"][0]["description"]

            forecast_rows.append({
                "Date": date_display,
                "Temperature": f"{temp:.1f}°C",
                "Feels Like": f"{feels:.1f}°C",
                "Condition": weather,
                "Description": description_day.title(),
                "Humidity": f"{humidity_day}%",
                "Wind": f"{wind:.1f} m/s"
            })


        forecast_df = pd.DataFrame(
            forecast_rows
        )


        # RETURN ALL RESULTS


        return (
            location,
            current_weather,
            humidity_text,
            wind_text,
            pressure_text,
            sunrise_text,
            sunset_text,
            icon_url,
            forecast_df
        )


    except requests.exceptions.ConnectionError:

        return (
            " Connection Error",
            "Please check your internet connection.",
            "",
            "",
            "",
            "",
            "",
            "",
            pd.DataFrame()
        )


    except requests.exceptions.Timeout:

        return (
            " Request Timeout",
            "The weather server took too long to respond.",
            "",
            "",
            "",
            "",
            "",
            "",
            pd.DataFrame()
        )


    except Exception as error:

        return (
            " Error",
            str(error),
            "",
            "",
            "",
            "",
            "",
            "",
            pd.DataFrame()
        )


# 3. GRADIO INTERFACE

with gr.Blocks(
    title="Weather Forecasting App"
) as app:

    gr.Markdown(
        """
        #  Weather Forecasting App

        ### Get current weather information and a 5-day forecast
        """
    )

    # SEARCH SECTION


    with gr.Row():

        city_input = gr.Textbox(
            label="Enter City",
            placeholder="Example: Chennai",
            scale=4
        )

        search_button = gr.Button(
            " Search Weather",
            variant="primary",
            scale=1
        )


    # LOCATION


    location_output = gr.Markdown(
        "##  Search for a city"
    )

    # CURRENT WEATHER


    with gr.Row():

        current_output = gr.Markdown(
            "###  Temperature\n\n--"
        )

        with gr.Column():

            humidity_output = gr.Markdown(
                " **Humidity:** --"
            )

            wind_output = gr.Markdown(
                " **Wind Speed:** --"
            )

            pressure_output = gr.Markdown(
                " **Pressure:** --"
            )

            sunrise_output = gr.Markdown(
                " **Sunrise:** --"
            )

            sunset_output = gr.Markdown(
                "**Sunset:** --"
            )


    # 5-DAY FORECAST

    gr.Markdown(
        "##  5-Day Forecast"
    )

    forecast_output = gr.Dataframe(
        headers=[
            "Date",
            "Temperature",
            "Feels Like",
            "Condition",
            "Description",
            "Humidity",
            "Wind"
        ],
        datatype=[
            "str",
            "str",
            "str",
            "str",
            "str",
            "str",
            "str"
        ],
        interactive=False
    )


    # FOOTER

    gr.Markdown(
        """
        ---
        **Powered by OpenWeather API 🌍**
        """
    )

    # BUTTON ACTION

    search_button.click(
        fn=get_weather,
        inputs=city_input,
        outputs=[
            location_output,
            current_output,
            humidity_output,
            wind_output,
            pressure_output,
            sunrise_output,
            sunset_output,
            gr.State(),
            forecast_output
        ]
    )

    # Press Enter to search
    city_input.submit(
        fn=get_weather,
        inputs=city_input,
        outputs=[
            location_output,
            current_output,
            humidity_output,
            wind_output,
            pressure_output,
            sunrise_output,
            sunset_output,
            gr.State(),
            forecast_output
        ]
    )

# 4. LAUNCH APP
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2e29b51aac14be251c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
